In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import importlib
import pandas as pd
from datetime import datetime
from tqdm import tqdm
from plotly.subplots import make_subplots
from plotly import tools
import plotly.offline as pyo
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import glob
from scipy import stats
import uproot
from ROOT import TFile, TEfficiency, TH1D, TGraphAsymmErrors, RDataFrame, TCanvas

c++: error: argument to '-isysroot' is missing (expected 1 value)
c++: error: no input files
ERROR in cling::CIFactory::createCI(): cannot extract standard library include paths!
Invoking:
  LC_ALL=C /Applications/Xcode.app/Contents/Developer/Toolchains/XcodeDefault.xctoolchain/usr/bin/c++ -isysroot;/Applications/Xcode.app/Contents/Developer/Platforms/MacOSX.platform/Developer/SDKs/MacOSX15.1.sdk   -xc++ -E -v /dev/null 2>&1 | sed -n -e '/^.include/,${' -e '/^ \/.*++/p' -e '}'
Results was:
c++: error: argument to '-isysroot' is missing (expected 1 value)
c++: error: no input files
With exit code 0


In [3]:
file = uproot.open("/Users/danielcarber/Documents/ICARUS/purity_mc.root")
print(file.keys())

['events;1', 'events/mc;1', 'events/mc/POT;4', 'events/mc/POT;3', 'events/mc/POT;2', 'events/mc/POT;1', 'events/mc/Livetime;4', 'events/mc/Livetime;3', 'events/mc/Livetime;2', 'events/mc/Livetime;1', 'events/mc/SelectedNu_Cuts;1', 'events/mc/SelectedCos_PhaseCuts;1', 'events/mc/Purity_PhaseCuts;81', 'events/mc/Purity_PhaseCuts;80', 'events/mc/Efficiency_PhaseCuts;1']


In [10]:
Nu_Purity = file['events/mc/Purity_PhaseCuts;80']
Cosmic_Purity = file['events/mc/SelectedCos_PhaseCuts;1']
Nu_Purity=Nu_Purity.arrays(library='pd')
Cosmic_Purity = Cosmic_Purity.arrays(library='pd')

print(Nu_Purity.keys())
print(sum(Nu_Purity['nu_id']>=0))
mask = (Nu_Purity['nu_id']>=0) & (Nu_Purity['catergory_topology']==0) &


Index(['all_1eNp_cut', 'category', 'category_topology', 'fiducial_cut',
       'flash_cut', 'nu_id', 'reco_NuMI_azi', 'reco_NuMI_polar', 'reco_Q',
       'reco_W', 'reco_dalphaT', 'reco_dpT', 'reco_dphiT',
       'reco_electron_NuMI_angle', 'reco_electron_axial_spread',
       'reco_electron_conv_dist', 'reco_electron_dedx',
       'reco_electron_dir_spread', 'reco_electron_energy',
       'reco_electron_pT_mag', 'reco_electron_polar',
       'reco_electron_primary_score', 'reco_electron_softmax',
       'reco_opening_angle', 'reco_proton_NuMI_angle', 'reco_proton_energy',
       'reco_proton_muon_softmax', 'reco_proton_pT_mag',
       'reco_proton_pion_softmax', 'reco_proton_polar',
       'reco_proton_primary_score', 'reco_proton_softmax', 'reco_topology',
       'reco_vertex_x', 'reco_vertex_y', 'reco_vertex_z',
       'reco_visible_energy', 'signal_1eNp', 'track_containment_cut',
       'true_NuMI_azi', 'true_NuMI_polar', 'true_Q', 'true_W', 'true_dalphaT',
       'true_dpT', 'true

In [149]:
quality_cuts =  (Nu_Purity['track_containment_cut'] ==1)

signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Containement cut: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Containement cut: 0.24%
Number of Signal and Total: 1938, 822424


In [147]:
quality_cuts =  (Nu_Purity['track_containment_cut'] ==1)&\
                (Nu_Purity['fiducial_cut'] ==1)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Fiducial cut: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Fiducial cut: 0.46%
Number of Signal and Total: 1820, 398750


In [146]:
quality_cuts = (Nu_Purity['flash_cut'] ==1)&\
                (Nu_Purity['track_containment_cut'] ==1)&\
                (Nu_Purity['fiducial_cut'] ==1)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Flash cut: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Flash cut: 2.26%
Number of Signal and Total: 1453, 64371


In [145]:
quality_cuts = (Nu_Purity['all_1eNp_cut'] ==1)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of 1eNp: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Directional Spread < 0.24: 79.86%
Number of Signal and Total: 1170, 1465


In [160]:
quality_cuts = ((Nu_Purity['all_1eNp_cut'] ==1)& (Nu_Purity['reco_electron_axial_spread'] > -0.01))
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Axial Spread > 0.02: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Axial Spread > 0.02: 80.54%
Number of Signal and Total: 1163, 1444


In [162]:
quality_cuts = (Nu_Purity['all_1eNp_cut'] ==1)&(Nu_Purity['reco_electron_axial_spread'] > -0.01)&(Nu_Purity['reco_electron_dir_spread'] < 0.23)
signal = Nu_Purity[quality_cuts]
mask  =  (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Directional Spread < 0.24: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Directional Spread < 0.24: 83.81%
Number of Signal and Total: 1144, 1365


In [163]:
quality_cuts =  (Nu_Purity['all_1eNp_cut'] ==1)&\
                (Nu_Purity['reco_electron_axial_spread'] > -0.01)&\
                (Nu_Purity['reco_electron_dir_spread'] < 0.23)&\
                (Nu_Purity['reco_electron_conv_dist'] <7.3)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Conversion Distance > 7.5: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Conversion Distance > 7.5: 85.30%
Number of Signal and Total: 1126, 1320


In [164]:
quality_cuts =  (Nu_Purity['all_1eNp_cut'] ==1)&\
                (Nu_Purity['reco_electron_axial_spread'] > -0.01) & \
                (Nu_Purity['reco_electron_dir_spread'] < 0.23)&\
                (Nu_Purity['reco_electron_conv_dist'] <7.3)&\
                (Nu_Purity['reco_proton_softmax'] >0.55)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Proton Softmax > 0.6: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Proton Softmax > 0.6: 86.32%
Number of Signal and Total: 1117, 1294


In [165]:
quality_cuts =  (Nu_Purity['all_1eNp_cut'] ==1)&\
                (Nu_Purity['reco_electron_axial_spread'] > -0.01) & \
                (Nu_Purity['reco_electron_dir_spread'] < 0.23)&\
                (Nu_Purity['reco_electron_conv_dist'] <7.3)&\
                (Nu_Purity['reco_proton_softmax'] >0.55)&\
                (Nu_Purity['reco_proton_muon_softmax'] <0.04)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Proton's Muon Softmax < 0,04: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Proton's Muon Softmax < 0,04: 87.12%
Number of Signal and Total: 1116, 1281


In [156]:
quality_cuts =  (Nu_Purity['all_1eNp_cut'] ==1)&\
                (Nu_Purity['reco_electron_axial_spread'] > 0.01) & \
                (Nu_Purity['reco_electron_dir_spread'] < 0.15)&\
                (Nu_Purity['reco_electron_conv_dist'] <5.5)&\
                (Nu_Purity['reco_proton_softmax'] >0.65)&\
                (Nu_Purity['reco_proton_muon_softmax'] <0.04)&\
                (Nu_Purity['reco_proton_pion_softmax'] <0.35)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Proton's Pion Softmax < 0.24: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Proton's Pion Softmax < 0.24: 88.59%
Number of Signal and Total: 1079, 1218


In [157]:
quality_cuts = (Nu_Purity['all_1eNp_cut'] ==1)&\
                (Nu_Purity['reco_electron_axial_spread'] > 0.01) & \
               (Nu_Purity['reco_electron_dir_spread'] < 0.15)&\
               (Nu_Purity['reco_electron_conv_dist'] <5.5)&\
               (Nu_Purity['reco_proton_softmax'] >0.65)&\
               (Nu_Purity['reco_proton_muon_softmax'] <0.04)&\
               (Nu_Purity['reco_proton_pion_softmax'] <0.35)&\
               (Nu_Purity['reco_electron_softmax'] <0.36)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Electron Softmax < 0.04: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Electron Softmax < 0.04: 88.93%
Number of Signal and Total: 1068, 1201


In [159]:
quality_cuts = (Nu_Purity['all_1eNp_cut'] ==1)&\
                (Nu_Purity['reco_electron_axial_spread'] > 0.01) & \
               (Nu_Purity['reco_electron_dir_spread'] < 0.15)&\
               (Nu_Purity['reco_electron_conv_dist'] <5.5)&\
               (Nu_Purity['reco_proton_softmax'] >0.65)&\
               (Nu_Purity['reco_proton_muon_softmax'] <0.04)&\
               (Nu_Purity['reco_proton_pion_softmax'] <0.35)&\
               (Nu_Purity['reco_electron_softmax'] <0.36)&\
               (Nu_Purity['reco_electron_primary_score'] >0.96)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Electron Primary Score > 0.97: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Electron Primary Score > 0.97: 89.53%
Number of Signal and Total: 1060, 1184
